# Fase 3 - Analitica Supervisada (Spark MLlib)

Regresion para predecir `Transfer Value` y clasificacion para el potencial de jovenes talentos (proyeccion via `Caps`).

In [1]:
import os
os.environ.setdefault('JAVA_HOME', '/opt/homebrew/opt/openjdk@17')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, GBTRegressor
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

spark = SparkSession.builder \
    .appName('ScoutingSupervised') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '8') \
    .getOrCreate()
spark.sparkContext.setLogLevel('WARN')

df = spark.read.parquet('../data/processed/players_clean.parquet')

attribute_cols = [
    'Acc','Wor','Vis','Thr','Tec','Tea','Tck','Str','Sta','TRO','Ref','Pun','Pos','Pen','Pas','Pac',
    '1v1','OtB','Habilidad_Natural','Mar','L Th','Lon','Ldr','Kic','Jum','Hea','Han','Fre','Fla','Fir','Fin','Ecc',
    'Dri','Det','Dec','Cro','Cor','Cnt','Cmp','Com','Cmd','Bra','Bal','Ant','Agi','Agg','Aer','Vers',
    'Temp','Spor','Prof','Pres','Loy','Inj Pr','Imp M','Dirt','Amb','Ada','Cons','Cont',
]
feature_cols = attribute_cols + ['Age', 'Height_cm', 'Weight_kg']
print(len(feature_cols), 'features')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/13 19:26:03 WARN Utils: Your hostname, MacBook-de-Rodrigo.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.2 instead (on interface en0)
26/07/13 19:26:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/13 19:26:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


63 features


## Parte A - Regresion: prediccion de Transfer Value

Se excluyen los jugadores `Not for Sale` (sin valor de mercado definido, ver Fase 1) del entrenamiento y evaluacion de la regresion.

In [2]:
df_reg = df.filter(F.col('Transfer_Value_num').isNotNull()).na.drop(subset=feature_cols)
print(df_reg.count())

train_reg, test_reg = df_reg.randomSplit([0.8, 0.2], seed=42)
print(train_reg.count(), test_reg.count())

26/07/13 19:26:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


89621


71793 17828


In [3]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features_raw')
scaler = StandardScaler(inputCol='features_raw', outputCol='features', withMean=True, withStd=True)

lr = LinearRegression(featuresCol='features', labelCol='Transfer_Value_num', maxIter=100)
gbt = GBTRegressor(featuresCol='features', labelCol='Transfer_Value_num', maxIter=50, maxDepth=5, seed=42)

pipeline_lr = Pipeline(stages=[assembler, scaler, lr])
pipeline_gbt = Pipeline(stages=[assembler, scaler, gbt])

reg_evaluator_rmse = RegressionEvaluator(labelCol='Transfer_Value_num', predictionCol='prediction', metricName='rmse')
reg_evaluator_mae = RegressionEvaluator(labelCol='Transfer_Value_num', predictionCol='prediction', metricName='mae')
reg_evaluator_r2 = RegressionEvaluator(labelCol='Transfer_Value_num', predictionCol='prediction', metricName='r2')

results_reg = {}
for name, pipeline in [('LinearRegression', pipeline_lr), ('GBTRegressor', pipeline_gbt)]:
    model = pipeline.fit(train_reg)
    preds_train = model.transform(train_reg)
    preds_test = model.transform(test_reg)
    results_reg[name] = {
        'rmse_train': reg_evaluator_rmse.evaluate(preds_train),
        'rmse_test': reg_evaluator_rmse.evaluate(preds_test),
        'mae_test': reg_evaluator_mae.evaluate(preds_test),
        'r2_test': reg_evaluator_r2.evaluate(preds_test),
        'model': model,
    }
    print(name, {k: v for k, v in results_reg[name].items() if k != 'model'})

26/07/13 19:26:11 WARN Instrumentation: [f8ec6566] regParam is zero, which might cause numerical instability and overfitting.


26/07/13 19:26:11 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


26/07/13 19:26:12 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


LinearRegression {'rmse_train': 6508012.371511441, 'rmse_test': 6057072.746065281, 'mae_test': 1915545.0114833785, 'r2_test': 0.12242899873086477}


GBTRegressor {'rmse_train': 2050683.8113262777, 'rmse_test': 4909783.967040158, 'mae_test': 848609.9273523052, 'r2_test': 0.42339095412033545}


`rmse_train` vs `rmse_test` cercanos indica que no hay overfitting severo. Si `rmse_train << rmse_test`, el modelo esta memorizando y se debe reducir `maxDepth` / aumentar regularizacion.

## Ajuste de hiperparametros con CrossValidator (GBTRegressor)

In [4]:
param_grid_gbt = (ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5, 7])
    .addGrid(gbt.maxIter, [30, 50])
    .build())

cv_gbt = CrossValidator(
    estimator=pipeline_gbt,
    estimatorParamMaps=param_grid_gbt,
    evaluator=reg_evaluator_rmse,
    numFolds=3,
    parallelism=2,
    seed=42,
)

cv_gbt_model = cv_gbt.fit(train_reg)
best_gbt_preds = cv_gbt_model.transform(test_reg)
print('RMSE test (mejor GBT via CV):', reg_evaluator_rmse.evaluate(best_gbt_preds))
print('R2 test (mejor GBT via CV):', reg_evaluator_r2.evaluate(best_gbt_preds))

26/07/13 19:27:43 WARN DAGScheduler: Broadcasting large task binary with size 1000.0 KiB
26/07/13 19:27:43 WARN DAGScheduler: Broadcasting large task binary with size 1008.5 KiB
26/07/13 19:27:43 WARN DAGScheduler: Broadcasting large task binary with size 1011.5 KiB


26/07/13 19:27:43 WARN DAGScheduler: Broadcasting large task binary with size 1012.0 KiB
26/07/13 19:27:43 WARN DAGScheduler: Broadcasting large task binary with size 1013.0 KiB
26/07/13 19:27:43 WARN DAGScheduler: Broadcasting large task binary with size 1013.8 KiB
26/07/13 19:27:43 WARN DAGScheduler: Broadcasting large task binary with size 1016.1 KiB
26/07/13 19:27:43 WARN DAGScheduler: Broadcasting large task binary with size 1020.6 KiB


26/07/13 19:27:43 WARN DAGScheduler: Broadcasting large task binary with size 1028.3 KiB
26/07/13 19:27:43 WARN DAGScheduler: Broadcasting large task binary with size 1030.2 KiB


26/07/13 19:27:44 WARN DAGScheduler: Broadcasting large task binary with size 1030.7 KiB
26/07/13 19:27:44 WARN DAGScheduler: Broadcasting large task binary with size 1031.7 KiB
26/07/13 19:27:44 WARN DAGScheduler: Broadcasting large task binary with size 1032.4 KiB
26/07/13 19:27:44 WARN DAGScheduler: Broadcasting large task binary with size 1034.1 KiB


26/07/13 19:27:44 WARN DAGScheduler: Broadcasting large task binary with size 1037.0 KiB
26/07/13 19:27:44 WARN DAGScheduler: Broadcasting large task binary with size 1042.6 KiB
26/07/13 19:27:44 WARN DAGScheduler: Broadcasting large task binary with size 1044.0 KiB
26/07/13 19:27:44 WARN DAGScheduler: Broadcasting large task binary with size 1044.5 KiB


...

26/07/13 19:29:32 WARN DAGScheduler: Broadcasting large task binary with size 1100.0 KiB
26/07/13 19:29:32 WARN DAGScheduler: Broadcasting large task binary with size 1103.9 KiB
26/07/13 19:29:32 WARN DAGScheduler: Broadcasting large task binary with size 1110.6 KiB
26/07/13 19:29:32 WARN DAGScheduler: Broadcasting large task binary with size 1112.5 KiB


26/07/13 19:29:32 WARN DAGScheduler: Broadcasting large task binary with size 1113.0 KiB
26/07/13 19:29:32 WARN DAGScheduler: Broadcasting large task binary with size 1114.0 KiB
26/07/13 19:29:32 WARN DAGScheduler: Broadcasting large task binary with size 1114.7 KiB
26/07/13 19:29:32 WARN DAGScheduler: Broadcasting large task binary with size 1117.1 KiB
26/07/13 19:29:32 WARN DAGScheduler: Broadcasting large task binary with size 1121.2 KiB


26/07/13 19:29:32 WARN DAGScheduler: Broadcasting large task binary with size 1128.2 KiB


RMSE test (mejor GBT via CV): 4622877.095701031


R2 test (mejor GBT via CV): 0.488811141844938


## Importancia de variables (GBTRegressor)

In [5]:
best_gbt_model = cv_gbt_model.bestModel.stages[-1]
importances = list(zip(feature_cols, best_gbt_model.featureImportances.toArray()))
importances.sort(key=lambda x: -x[1])
for feat, imp in importances[:15]:
    print(f"{feat}: {imp:.4f}")

Pac: 0.1443
Cmp: 0.0873
Tec: 0.0684
Dec: 0.0621
Ant: 0.0596
Acc: 0.0481
Sta: 0.0479
Fir: 0.0379
Hea: 0.0326
Bal: 0.0320
Vis: 0.0262
Pos: 0.0245
Tck: 0.0220
Cro: 0.0215
Dri: 0.0202


## Parte B - Clasificacion: potencial de jovenes talentos

Target: si el jugador ha sido convocado a su seleccion (`Caps > 0`), usando solo atributos de habilidad (sin `Transfer Value`, que dependeria del resultado y causaria data leakage). El modelo se entrena con todos los jugadores y se aplica luego sobre los menores de 21 anios como 'proyeccion de potencial'.

In [6]:
df_clf = df.na.drop(subset=feature_cols).withColumn(
    'label', (F.coalesce(F.col('Caps'), F.lit(0)) > 0).cast('double')
)
df_clf.groupBy('label').count().show()

train_clf, test_clf = df_clf.randomSplit([0.8, 0.2], seed=42)

+-----+-----+
|label|count|
+-----+-----+
|  1.0| 7789|
|  0.0|83883|
+-----+-----+



In [7]:
assembler_clf = VectorAssembler(inputCols=feature_cols, outputCol='features_raw')
scaler_clf = StandardScaler(inputCol='features_raw', outputCol='features', withMean=True, withStd=True)

log_reg = LogisticRegression(featuresCol='features', labelCol='label', maxIter=100, regParam=0.01)
rf_clf = RandomForestClassifier(featuresCol='features', labelCol='label', maxDepth=6, numTrees=100, seed=42)

pipeline_logreg = Pipeline(stages=[assembler_clf, scaler_clf, log_reg])
pipeline_rf = Pipeline(stages=[assembler_clf, scaler_clf, rf_clf])

f1_evaluator = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1')
auc_evaluator = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC')

results_clf = {}
for name, pipeline in [('LogisticRegression', pipeline_logreg), ('RandomForest', pipeline_rf)]:
    model = pipeline.fit(train_clf)
    preds_train = model.transform(train_clf)
    preds_test = model.transform(test_clf)
    results_clf[name] = {
        'f1_train': f1_evaluator.evaluate(preds_train),
        'f1_test': f1_evaluator.evaluate(preds_test),
        'auc_test': auc_evaluator.evaluate(preds_test),
        'model': model,
    }
    print(name, {k: v for k, v in results_clf[name].items() if k != 'model'})

LogisticRegression {'f1_train': 0.9025157573729025, 'f1_test': 0.9051289702339715, 'auc_test': 0.8553537693019319}


26/07/13 19:29:57 WARN DAGScheduler: Broadcasting large task binary with size 1198.3 KiB


26/07/13 19:29:59 WARN DAGScheduler: Broadcasting large task binary with size 1073.4 KiB


26/07/13 19:30:00 WARN DAGScheduler: Broadcasting large task binary with size 1073.3 KiB


26/07/13 19:30:00 WARN DAGScheduler: Broadcasting large task binary with size 1060.9 KiB


RandomForest {'f1_train': 0.8899363423033213, 'f1_test': 0.8914261316650202, 'auc_test': 0.849444640145735}


## Ajuste de hiperparametros con CrossValidator (RandomForest) y control de overfitting

In [8]:
param_grid_rf = (ParamGridBuilder()
    .addGrid(rf_clf.maxDepth, [4, 6, 8])
    .addGrid(rf_clf.numTrees, [50, 100])
    .build())

cv_rf = CrossValidator(
    estimator=pipeline_rf,
    estimatorParamMaps=param_grid_rf,
    evaluator=f1_evaluator,
    numFolds=3,
    parallelism=2,
    seed=42,
)

cv_rf_model = cv_rf.fit(train_clf)
preds_train_best = cv_rf_model.transform(train_clf)
preds_test_best = cv_rf_model.transform(test_clf)
print('F1 train (mejor RF via CV):', f1_evaluator.evaluate(preds_train_best))
print('F1 test (mejor RF via CV):', f1_evaluator.evaluate(preds_test_best))
print('AUC test (mejor RF via CV):', auc_evaluator.evaluate(preds_test_best))

26/07/13 19:30:10 WARN DAGScheduler: Broadcasting large task binary with size 1229.6 KiB


26/07/13 19:30:12 WARN DAGScheduler: Broadcasting large task binary with size 1129.5 KiB


26/07/13 19:30:13 WARN DAGScheduler: Broadcasting large task binary with size 1184.8 KiB


26/07/13 19:30:14 WARN DAGScheduler: Broadcasting large task binary with size 1901.7 KiB


26/07/13 19:30:16 WARN DAGScheduler: Broadcasting large task binary with size 1550.1 KiB


...

F1 train (mejor RF via CV): 0.9024744034751069


26/07/13 19:31:07 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB


F1 test (mejor RF via CV): 0.896950856694312


26/07/13 19:31:08 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB


AUC test (mejor RF via CV): 0.8606798551905522


## Importancia de variables (RandomForest)

In [9]:
best_rf_model = cv_rf_model.bestModel.stages[-1]
importances_clf = list(zip(feature_cols, best_rf_model.featureImportances.toArray()))
importances_clf.sort(key=lambda x: -x[1])
for feat, imp in importances_clf[:15]:
    print(f"{feat}: {imp:.4f}")

Cmp: 0.0903
Bal: 0.0843
Amb: 0.0695
Ant: 0.0693
Age: 0.0490
Bra: 0.0380
Pen: 0.0366
Cnt: 0.0326
Str: 0.0315
Prof: 0.0297
Tea: 0.0295
Sta: 0.0274
Pac: 0.0256
OtB: 0.0221
Wor: 0.0214


## Aplicacion: proyeccion de potencial en jovenes talentos (<= 21 anios)

In [10]:
from pyspark.ml.functions import vector_to_array

jovenes = df_clf.filter(F.col('Age') <= 21)
proyeccion = cv_rf_model.transform(jovenes).withColumn('prob_potencial', vector_to_array('probability')[1])
proyeccion.select('Name', 'Age', 'Position', 'Club', 'label', 'prediction', 'prob_potencial') \
    .orderBy(F.desc('prob_potencial')) \
    .show(15, truncate=False)

26/07/13 19:31:09 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB


+--------------------+---+----------------+-----------------+-----+----------+------------------+
|Name                |Age|Position        |Club             |label|prediction|prob_potencial    |
+--------------------+---+----------------+-----------------+-----+----------+------------------+
|Jude Bellingham     |18 |DM, M (C)       |Borussia Dortmund|1.0  |1.0       |0.8441574634335655|
|Jamal Musiala       |19 |M (C), AM (RLC) |FC Bayern        |1.0  |1.0       |0.8300263544668735|
|Vinícius Júnior     |21 |AM (RL)         |R. Madrid        |1.0  |1.0       |0.824013036846368 |
|Bukayo Saka         |20 |AM (RL)         |Arsenal          |1.0  |1.0       |0.820011840647906 |
|Erling Haaland      |21 |ST (C)          |Man City         |1.0  |1.0       |0.8149923344357741|
|Charles De Ketelaere|21 |M/AM (C), ST (C)|Milan            |1.0  |1.0       |0.7571053497851453|
|Emile Smith Rowe    |21 |AM (RLC)        |Arsenal          |1.0  |1.0       |0.7479316252266981|
|Gavi               